In [1]:
# imports
import pandas as pd
import numpy as np
import plotly_express as px
import plotly.graph_objects as go

# Exploratory Data Analysis on GID Open Data

## Loading and Cleaning Data

In [2]:
# Defines relevant columns and scope of data
cols = [
    'service_request_id', 'service_request_parent_id', 'date_requested', 'case_age_days',
    'case_record_type', 'service_name', 'service_name_detail', 'date_closed', 'status', 
    'lat', 'lng', 'council_district', 'case_origin', 'referred', 'public_description'
    ]

years = [2023, 2024, 2025, 2026]

In [3]:
# Load datasets from url
for y in years:
    url = f'https://seshat.datasd.org/get_it_done_reports/get_it_done_requests_closed_{y}_datasd.csv'
    var = f'datasd_{y}'
    globals()[var] = pd.read_csv(url, usecols=cols)

In [4]:
datasd = pd.concat([globals()[f'datasd_{y}'] for y in years], ignore_index=True)

In [5]:
datasd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1596579 entries, 0 to 1596578
Data columns (total 15 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   service_request_id         1596579 non-null  int64  
 1   service_request_parent_id  213917 non-null   float64
 2   date_requested             1596579 non-null  object 
 3   case_age_days              1596579 non-null  int64  
 4   case_record_type           1596579 non-null  object 
 5   service_name               1589005 non-null  object 
 6   service_name_detail        929956 non-null   object 
 7   date_closed                1596579 non-null  object 
 8   status                     1596579 non-null  object 
 9   lat                        1589522 non-null  float64
 10  lng                        1589522 non-null  float64
 11  council_district           1586670 non-null  float64
 12  case_origin                1596412 non-null  object 
 13  referred    

In [6]:
# Remove duplicate requests
datasd = datasd[datasd['service_request_parent_id'].isna()]

datasd.drop(columns=['service_request_parent_id'], inplace=True)

# Count total number of reports again from the deduplicated data
datasd['service_request_id'].nunique()

1382653

In [7]:
datasd[datasd.duplicated(subset=['service_request_id'], keep=False)].sort_values('service_request_id').head(16)

,service_request_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,lat,lng,council_district,case_origin,referred,public_description
1138715,4814585,2024-06-27 07:34:00.000,267,DSD,STRO Violation,NaN,2025-03-21,Closed,NaN,NaN,NaN,Email,NaN,NaN
1378970,4814585,2024-06-27 07:34:00.000,769,DSD,STRO Violation,NaN,2026-08-05,Closed,NaN,NaN,NaN,Email,NaN,NaN
817414,5038053,2024-12-13 14:01:00.000,3,DSD,Other,NaN,2024-12-16,Closed,32.707238,-117.079204,4.0,Mobile,NaN,"Our neighbor at 5480 imperial Ave. SAN Diego, ..."
1428597,5038053,2024-12-13 14:01:00.000,615,DSD,Other,NaN,2026-08-20,Closed,32.707238,-117.079204,4.0,Mobile,NaN,"This neighbor at 5480 imperial Ave. SAN Diego,..."
942414,5310443,2025-07-28 21:38:00.000,72,Storm Water Code Enforcement,Other,NaN,2025-10-08,Closed,32.707909,-117.122010,9.0,Mobile,NaN,Illegal dumping from trailer. Dumping waste in...
1513391,5310443,2025-07-28 21:38:00.000,346,Storm Water Code Enforcement,Other,NaN,2026-07-09,Closed,32.707909,-117.122010,9.0,Mobile,NaN,Illegal dumping from trailer. Dumping waste in...
1077335,5352404,2025-08-28 10:42:00.000,39,ESD Complaint/Report,Environmental Services Code Compliance,Container Left Out,2025-10-06,Closed,32.769004,-117.186602,7.0,Mobile,NaN,5958 Lauretta st
1527370,5352404,2025-08-28 10:42:00.000,340,ESD Complaint/Report,Environmental Services Code Compliance,Container Left Out,2026-08-03,Closed,32.768874,-117.186395,7.0,Mobile,NaN,5958 Lauretta st
1114902,5367887,2025-09-08 14:10:00.000,43,ESD Complaint/Report,Waste on Private Property,NaN,2025-10-21,Closed,32.740817,-117.057803,4.0,Mobile,NaN,Appliances on the street in view.
1528053,5367887,2025-09-08 14:10:00.000,324,ESD Complaint/Report,Waste on Private Property,NaN,2026-07-29,Closed,32.740799,-117.057862,4.0,Mobile,NaN,Appliances on the street in view.


In [8]:
datasd_duplicates = datasd[datasd.duplicated(subset=['service_request_id'], keep=False)].sort_values('service_request_id')
datasd = datasd.drop_duplicates(subset=['service_request_id'], keep='last')

In [9]:
# Converting values to correct dtypes
datasd['date_requested'] = pd.to_datetime(datasd['date_requested'])
datasd['date_closed'] = pd.to_datetime(datasd['date_closed'])
datasd['council_district'] = datasd['council_district'].astype('category')

In [10]:
datasd.loc[datasd['referred'].notna(), 'referred'].head()

20    This report has been referred to Police ? Cent...
23    This report has been referred to Field Enginee...
39    This report has been referred to Water Repair ...
40    This report has been referred to Water Repair ...
71    This report has been referred to Parking Meter...
Name: referred, dtype: object

In [11]:
# Inplace function(s) and code to clean up referred column. Based on the frequency of referrals made
def categorize_referrals(txt):
    txt = str(txt).lower()
    if 'sdge' in txt or 'sdg&e' in txt:
        return 'sdg&e'
    if 'cox.com' in txt:
        return 'cox communications'
    if 'charter.com' in txt or 'spectrum' in txt:
        return 'spectrum'
    if 'att' in txt or 'at&t' in txt:
        return 'at&t'
    if 'cleanharbors' in txt:
        return 'clean harbors sanitation'
    if 'wm.com' in txt or 'waste management' in txt:
        return 'waste management'
    if 'republicservices' in txt or 'republic-services' in txt:
        return 'republic services'
    if 'edcodisposal' in txt:
        return 'edco disposal'
    if 'sdmts' in txt:
        return 'mts'
    if 'nctd.org' in txt:
        return 'nctd'
    if 'bnsf' in txt:
        return 'bnsf railroad'
    if 'caltrans' in txt or 'dot.ca.gov' in txt:
        return 'caltrans'
    if 'portofsandiego' in txt or 'port of san diego' in txt:
        return 'port of sd'
    if 'san.org' in txt:
        return 'sd international airport'
    if 'sandi.net' in txt:
        return 'sd unified school district'
    if 'ucsd.edu' in txt:
        return 'ucsd'
    if 'sdsu.edu' in txt:
        return 'sdsu'
    if 'sdccd.edu' in txt:
        return 'sdccd'
    if 'chulavistaca.gov' in txt:
        return 'city of chula vista'
    if 'sdcounty.ca.gov' in txt:
        return 'san diego county'
    if 'parks.ca.gov' in txt or 'wildlife.ca.gov' in txt:
        return 'california state parks/wildlife'
    if 'usps' in txt:
        return 'usps'
    if 'livelmh' in txt:
        return 'liberty military housing'
    if 'improvedtsd' in txt or 'ballparkdistrict' in txt or 'downtownsandiego.org' in txt:
        return 'downtown clean & safe / ballpark'
    if 'sdhumane' in txt:
        return 'sd humane society'
    if 'vector' in txt:
        return 'county vector control'
    if 'rms' in txt or 'retailsolutions' in txt or 'retailms.net' in txt:
        return 'retail solutions (rms)'
    if 'smartcart' in txt:
        return 'smart cart'
    if 'clearchannel' in txt:
        return 'clear channel'
    if 'outfront.com' in txt:
        return 'outfront media'
    if 'engineering' in txt:
        return 'field engineering'
    if 'facilitiesmaint' in txt:
        return 'facilities maintenance'
    if 'treepermit' in txt:
        return 'tree permits'
    if 'publicliability' in txt or 'riskmanagement' in txt:
        return 'risk management / public liability'
    if 'specialevents' in txt:
        return 'special events'
    if 'realestate' in txt:
        return 'real estate assets'
    if 'sdpd' in txt or 'police' in txt:
        return 'sdpd'
    if 'sdfd' in txt or 'fire' in txt:
        return 'sdfd'
    if 'wcm_gid' in txt or 'water' in txt:
        return 'water / wastewater (pud)'
    if 'jkafulides' in txt or 'parking' in txt:
        return 'parking meter repair'
    if 'sdlg' in txt or 'lifeguard' in txt:
        return 'lifeguards'
    if 'fit_dsd' in txt or 'development services' in txt:
        return 'development services (dsd)'
    if 'sdbusiness' in txt:
        return 'sd business / economic dev'
    if 'street division' in txt:
        return 'street division'
    if 't-row code compliance' in txt:
        return 't-row code compliance'     
    if 'sanitization contractor' in txt or 'environmental services' in txt or 'esd' in txt:
        return 'environmental services (esd)'
    if 'parks intake' in txt:
        return 'parks & recreation'
    if 'testing only' in txt:
        return 'test/invalid'
    if 'null' in txt:
        return 'error'
    return txt

def clean_referrals(df):
    df['referred'] = df['referred'].apply(categorize_referrals)
    # If the referral is not a specific organization (is an email), extract the email:
    df.loc[df['referred'].str.contains('@'), 'referred'] = df.loc[df['referred'].str.contains('@'), 'referred'].str.extract(r'([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})')[0].str.lower()
    df.loc[df['referred'] == 'nan', 'referred'] = np.nan

In [12]:
clean_referrals(datasd)

datasd.loc[datasd['referred'].notna(), 'referred'].head()

20                        sdpd
23           field engineering
39    water / wastewater (pud)
40    water / wastewater (pud)
71        parking meter repair
Name: referred, dtype: object

In [13]:
datasd.shape

(1382653, 14)

In [14]:
datasd.head()

,service_request_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,lat,lng,council_district,case_origin,referred,public_description
0,4470480,2023-10-20 08:24:00,0,ESD Complaint/Report,Illegal Dumping,Illegal Dumping,2023-10-20,Closed,32.702221,-117.134512,8.0,Worker App,NaN,NaN
1,4470495,2023-10-20 08:28:00,0,ESD Complaint/Report,Illegal Dumping,Illegal Dumping,2023-10-20,Closed,32.704880,-117.133812,8.0,Worker App,NaN,NaN
2,4470497,2023-10-20 08:28:00,3,Parking,Parking Violation,Red Zone,2023-10-23,Closed,33.027348,-117.077324,5.0,Mobile,NaN,Red zone n fire hydrate parking? HIS front tir...
4,4470521,2023-10-20 08:33:00,0,TSW,Graffiti - Public,NaN,2023-10-20,Closed,32.719937,-117.171557,3.0,Crew/Self Generated,NaN,NaN
5,4470529,2023-10-20 08:35:00,0,ESD Complaint/Report,Illegal Dumping,Illegal Dumping,2023-10-20,Closed,32.759370,-117.114687,9.0,Worker App,NaN,NaN


In [15]:
gid_open = pd.read_csv('https://seshat.datasd.org/get_it_done_reports/get_it_done_requests_open_datasd.csv', usecols=cols)

# Remove duplicate requests
gid_open = gid_open[gid_open['service_request_parent_id'].isna()]

gid_open.drop(columns=['service_request_parent_id'], inplace=True)

In [16]:
gid_open[gid_open.duplicated(subset=['service_request_id'], keep=False)].sort_values('service_request_id').head(10)

,service_request_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,lat,lng,council_district,case_origin,referred,public_description


In [17]:
gid_open['date_requested'] = pd.to_datetime(gid_open['date_requested'])
gid_open['council_district'] = gid_open['council_district'].astype('category')
clean_referrals(gid_open)

In [18]:
gid_open.shape

(62602, 14)

In [19]:
gid_open.head()

,service_request_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,lat,lng,council_district,case_origin,referred,public_description
0,5203672,2025-04-30 18:04:26,553,TSW,ROW Maintenance,TRAFFIC DEVICES,NaN,In Process,32.711242,-117.164650,3.0,Mobile,NaN,This âRoad Work Aheadâ sign has been up si...
3,5203720,2025-04-30 18:57:26,553,DSD,Development Services - Code Enforcement,DSD BLUE,NaN,In Process,32.554129,-117.045989,8.0,Mobile,NaN,Building without permits
4,5203756,2025-04-30 19:44:14,553,TSW,ROW Maintenance,SIGN - MAINT - STREET NAME,NaN,In Process,33.019412,-117.120070,5.0,Web,NaN,When turning from Camino del Norte;Right onto...
9,5203923,2025-05-01 04:40:59,552,DSD,Development Services - Code Enforcement,DSD BLUE,NaN,In Process,32.758679,-117.138277,3.0,Mobile,NaN,Property owner installed industrial flood ligh...
10,5203933,2025-05-01 05:35:15,552,TSW,Pavement Maintenance,SURFACE FLOW ISSUE,NaN,In Process,32.893802,-117.118120,6.0,Mobile,NaN,Deep standing water for over a week again


In [20]:
df = pd.concat([datasd, gid_open], ignore_index=True)

/var/folders/sp/1qg6hw2572g23yxdgsh3gbkm0000gn/T/ipykernel_2854/4261117055.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([datasd, gid_open], ignore_index=True)


In [21]:
df[df.duplicated(subset=['service_request_id'], keep=False)].sort_values('service_request_id').head(10)

,service_request_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,lat,lng,council_district,case_origin,referred,public_description
759134,2416658,2018-10-05 11:54:00,2387,DSD,Development Services - Code Enforcement,Quality of Life Issues,2025-04-18,Closed,32.707764,-117.070610,4.0,Mobile,NaN,"Misc couches, cabinets and random items in fro..."
1411205,2416658,2018-10-05 11:54:01,2952,DSD,Development Services - Code Enforcement,Quality of Life Issues,NaT,New,32.707764,-117.070610,4.0,Mobile,NaN,"Misc couches, cabinets and random items in fro..."
397795,4278707,2023-06-13 07:21:00,482,TSW ROW,Vegetation Encroachment,NaN,2024-10-07,Referred,32.708625,-117.073520,4.0,Mobile,real estate assets,Over grown side walk bushes
1432213,4278707,2023-06-13 07:21:51,1240,TSW,Stormwater,CHANNEL,NaT,In Process,32.708625,-117.073520,4.0,Mobile,real estate assets,Over grown side walk bushes
836731,4600153,2024-01-26 11:02:00,629,TSW ROW,Right-of-Way Code Enforcement,Trenches,2025-10-16,Closed,32.871820,-117.217867,6.0,Mobile,NaN,Subgrade Issue. Pavement lifted 4-6inches. La ...
1435208,4600153,2024-01-26 11:02:24,1013,TSW ROW,Right-of-Way Code Enforcement,Trenches,NaT,In Process,32.871820,-117.217867,6.0,Mobile,NaN,Subgrade Issue. Pavement lifted 4-6inches. La ...
839422,4744770,2024-05-08 16:39:00,342,DSD,Development Services - Code Enforcement,NaN,2025-04-15,Closed,32.699908,-117.097744,4.0,Mobile,NaN,The home next to mine located at 529 S 45th St...
1436947,4744770,2024-05-08 16:39:32,910,DSD,Building and Land Use Enforcement,NaN,NaT,New,32.699908,-117.097744,4.0,Mobile,NaN,The home located at 529 S 45th Street SDCA 921...
525618,4915091,2024-09-04 10:26:00,33,TSW ROW,Vegetation Encroachment,NaN,2024-10-07,Referred,32.708541,-117.073692,4.0,Mobile,real estate assets,Overgrown vegetation
1439021,4915091,2024-09-04 10:26:49,791,TSW,Stormwater,CHANNEL,NaT,In Process,32.708541,-117.073692,4.0,Mobile,real estate assets,Overgrown vegetation


In [22]:
df_duplicates = df[df.duplicated(subset=['service_request_id'], keep=False)].sort_values('service_request_id')
df = df.drop_duplicates(subset=['service_request_id'], keep='last')

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1445242 entries, 0 to 1445254
Data columns (total 14 columns):
 #   Column               Non-Null Count    Dtype         
---  ------               --------------    -----         
 0   service_request_id   1445242 non-null  int64         
 1   date_requested       1445242 non-null  datetime64[ns]
 2   case_age_days        1445242 non-null  int64         
 3   case_record_type     1445242 non-null  object        
 4   service_name         1436939 non-null  object        
 5   service_name_detail  892358 non-null   object        
 6   date_closed          1382640 non-null  datetime64[ns]
 7   status               1445242 non-null  object        
 8   lat                  1437338 non-null  float64       
 9   lng                  1437338 non-null  float64       
 10  council_district     1434454 non-null  category      
 11  case_origin          1445074 non-null  object        
 12  referred             108762 non-null   object        
 13  pu

In [24]:
df.head()

,service_request_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,lat,lng,council_district,case_origin,referred,public_description
0,4470480,2023-10-20 08:24:00,0,ESD Complaint/Report,Illegal Dumping,Illegal Dumping,2023-10-20,Closed,32.702221,-117.134512,8.0,Worker App,NaN,NaN
1,4470495,2023-10-20 08:28:00,0,ESD Complaint/Report,Illegal Dumping,Illegal Dumping,2023-10-20,Closed,32.704880,-117.133812,8.0,Worker App,NaN,NaN
2,4470497,2023-10-20 08:28:00,3,Parking,Parking Violation,Red Zone,2023-10-23,Closed,33.027348,-117.077324,5.0,Mobile,NaN,Red zone n fire hydrate parking? HIS front tir...
3,4470521,2023-10-20 08:33:00,0,TSW,Graffiti - Public,NaN,2023-10-20,Closed,32.719937,-117.171557,3.0,Crew/Self Generated,NaN,NaN
4,4470529,2023-10-20 08:35:00,0,ESD Complaint/Report,Illegal Dumping,Illegal Dumping,2023-10-20,Closed,32.759370,-117.114687,9.0,Worker App,NaN,NaN


## EDA

### What is the current workload and system?

In [25]:
N = 15 # global variable

In [26]:
complete_yrs = df[df['date_requested'].dt.year >= min(years)].copy()
print('Total Number of Distinct Services Offered Every Year')
complete_yrs['year_opened'] = complete_yrs['date_requested'].dt.year
complete_yrs.groupby('year_opened')[['service_name', 'service_name_detail']].nunique()

Total Number of Distinct Services Offered Every Year


,service_name,service_name_detail
year_opened,,
2023,40,217
2024,40,253
2025,40,254
2026,38,263


In [27]:
intake = complete_yrs.groupby(complete_yrs['date_requested'].dt.to_period('M')).size().reset_index(name='intake')
intake['month'] = intake['date_requested'].dt.to_timestamp()

outflow = complete_yrs.groupby(complete_yrs['date_closed'].dt.to_period('M')).size().reset_index(name='outflow')
outflow['month'] = outflow['date_closed'].dt.to_timestamp()

req_vol_df = pd.merge(intake[['month', 'intake']], outflow[['month', 'outflow']], on='month', how='outer').fillna(0)
req_vol_df.sort_values('month', inplace=True)

# Filter out incomplete month (October 2026)
req_vol_df = req_vol_df[req_vol_df['month'] < '2026-10-01']

In [28]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=req_vol_df['month'], y=req_vol_df['intake'], mode='lines', name='Reports Received'))
fig.add_trace(go.Scatter(x=req_vol_df['month'], y=req_vol_df['outflow'], mode='lines', name='Reports Completed'))
fig.update_layout(
    title=f"Monthly Intake and Outflow ({min(years)}-{max(years)})<br><span style='font-size: 13px; color: gray;'>Outflow does not necessarily mean any work occurred.</span>",
    xaxis_title='Date (Month Yr.)',
    yaxis_title='Total Reports',
    hovermode='x unified',
    xaxis={'tickformat': '%b %Y', 'hoverformat': '%b %Y'}
    )

In [29]:
referrals = datasd\
    .groupby('service_name')['status']\
    .apply(
        lambda t: (t == 'Referred').mean().round(4) * 100
        )\
    .reset_index(name='proportion')\
    .sort_values('proportion', ascending=False)
px.bar(
    referrals[:N],
    x='service_name',
    y='proportion',
    title=f'Highest {N} Proportion of Referrals by Service ({min(years)}-{max(years)})',
    labels={'service_name': 'Service Category', 'proportion': 'Proportion'}
    )

In [30]:
med_rsps_days = datasd.groupby('service_name')['case_age_days'].median()
highest_med_days = med_rsps_days.sort_values(ascending=False).reset_index()[:N]
fig = px.bar(
    highest_med_days.sort_values('case_age_days'),
    y='service_name',
    x='case_age_days',
    text_auto='.f',
    title=f'Top {N} Services With Highest Median Time (Days) to Respond',
    labels={'case_age_days': 'Median Response Time (Days)', 'service_name': 'Service Category'}
    )
fig.update_traces(hoverinfo='none', hovertemplate=None, textposition='outside')

In [31]:
med_rsps_days = datasd.groupby('service_name')['case_age_days'].median()
lowest_med_days = med_rsps_days.sort_values().reset_index()[:N]
fig = px.bar(lowest_med_days.sort_values('case_age_days', ascending=False), y='service_name', x='case_age_days', text_auto='.f')
fig.update_traces(hoverinfo='none', hovertemplate=None, textposition='outside')

Given that `datasd` contains closed cases, these visuals show what the City prioritizes (and responds to efficiently) and where responses might be delayed. Notably, there is a common theme among services with high median case ages: private property or street/traffic-related services.

Does `case_age_days` differ based on `service`?

In [32]:
from scipy.stats import kruskal

# Use closed reports with valid case durations
kruskalwallis_data = datasd[
    datasd['service_name'].notna()
    & datasd['case_age_days'].notna()
    & (datasd['case_age_days'] >= 0)
].copy()

# Select services with sufficient observations
top_services = (
    kruskalwallis_data['service_name']
    .value_counts()
    .loc[lambda counts: counts > 30]
    .index
)

groups = [
    kruskalwallis_data.loc[
        kruskalwallis_data['service_name'] == service,
        'case_age_days'
    ].to_numpy()
    for service in top_services
    ]

print(f'Number of distinct services kept: {top_services.nunique()}\n')

statistic, p_value = kruskal(*groups)

print(f'Kruskal-Wallis statistic: {statistic:.3f}')
print(f'p-value: {p_value:.4e}')

Number of distinct services kept: 39

Kruskal-Wallis statistic: 524891.796
p-value: 0.0000e+00


Is `referred` dependent of `service` category?

In [33]:
from scipy.stats import chi2_contingency

chi2_data = datasd[datasd['service_name'].notna()].copy()
chi2_data['is_referred'] = chi2_data['referred'].notna()

contingency_table = pd.crosstab(chi2_data['service_name'], chi2_data['is_referred'])

print(f'First 5 rows of contingency table:\n{contingency_table.head()}\n')

chi2_stat, p_val, dof, expected = chi2_contingency(contingency_table)

print(f'Chi-square Statistic: {chi2_stat:.2f}')
print(f'P-value: {p_val:.4e}')

First 5 rows of contingency table:
is_referred                        False  True 
service_name                                   
ADA Compliance                         1      0
Bicycle-Related Issue                  4      1
Biohazard                              7    479
Building and Land Use Enforcement   1445    553
Container Request                      3      0

Chi-square Statistic: 331221.08
P-value: 0.0000e+00


### Where might a computer vision model improve the current system?
<!-- 2. What is the current workload and system? -->

In [34]:
# What is the most common service still unaddressed from each year? (as of October 2026)
open_servs = pd.crosstab(index=gid_open['service_name'], columns=gid_open['date_requested'].dt.year).idxmax()
open_servs 

date_requested
2016        Pavement Maintenance
2017       Sidewalk Repair Issue
2018       Sidewalk Repair Issue
2019       Sidewalk Repair Issue
2020       Sidewalk Repair Issue
2021       Sidewalk Repair Issue
2022       Sidewalk Repair Issue
2023       Sidewalk Repair Issue
2024       Sidewalk Repair Issue
2025    Street Light Maintenance
2026          Parking - 72-Hours
dtype: object

In [35]:
open_reports = gid_open.copy()
open_reports['year'] = open_reports['date_requested'].dt.year
open_reports = open_reports[open_reports['year'] != 2026]
open_reports_data = open_reports.groupby(['year', 'service_name']).size().reset_index(name='count')
px.bar(
    open_reports_data[open_reports_data['service_name'].isin(open_reports['service_name'].value_counts().index[:5])],
    x='year',
    y='count',
    color='service_name',
    barmode='group',
    title=f"Breakdown of Most Common Open Reports by Year Originally Reported",
    labels={'year': 'Year Opened', 'service_name': 'Service Category', 'count': 'Total Reports'},
    )

`Sidewalk Repair Issue` cases are consistenly the largest amount of open reports in earlier years, followed by `Pavement Maintenance`.


<!-- from docx import Document
doc = Document('ROW Support Matrix.docx').tables[0]

data = [[cell.text.strip() for cell in row.cells] for row in doc.rows]

matrix = pd.DataFrame(data[1:], columns=data[0])
matrix = matrix.replace('', np.nan) -->

From before, we saw that `Other` had the second highest proportion of referrals (after `Biohazard`). The most common referral is SDPD (San Diego Police Department); The average response time for these referrals is 0.00

In [36]:
datasd.loc[datasd['service_name'] == 'Other', 'service_name_detail'].unique()

array([nan], dtype=object)

In [37]:
other_referrals = datasd.loc[(datasd['service_name'] == 'Other'), 'referred'].value_counts()
px.bar(other_referrals[:N])

In [38]:
px.box(datasd.loc[(datasd['service_name'] == 'Other') & (datasd['referred'].notna()) & (datasd['referred'].isin(other_referrals[:N].index)), 'case_age_days'], orientation='h')

**The existing Get It Done system processes a substantial volume of public requests, but the available open data does not establish how accurately or efficiently requests are initially routed. My understanding is that City intake staff manually use the ROW Matrix Service to help determine the appropriate service category. An image-assisted classification or validation tool could complement this workflow by providing additional evidence from submitted images, descriptions, and location data when assigning service categories.**

### What can be measured operationally?

This data does not provide enough information about how misrouted cases are dealt with, such as the timeframe between the initial routing and the correction.

What we currently have is the number of days taken for a case's status to update, i.e. from `Open` to `In Process`, and even then it only contains the most recent status update. So while `case_age_days` can tell us how long the process takes for cases and if there are any systemic problems like certain services seeing overly delayed responses, we cannot measure the operational impacts at least in terms of delays.
 
Lastly, the duplicated cases from before indicate some intervention, like updating a service category. But these appear to be errors when updating/transfering the data to the open portal.

In [39]:
datasd_duplicates.head(16)

,service_request_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,lat,lng,council_district,case_origin,referred,public_description
1138715,4814585,2024-06-27 07:34:00.000,267,DSD,STRO Violation,NaN,2025-03-21,Closed,NaN,NaN,NaN,Email,NaN,NaN
1378970,4814585,2024-06-27 07:34:00.000,769,DSD,STRO Violation,NaN,2026-08-05,Closed,NaN,NaN,NaN,Email,NaN,NaN
817414,5038053,2024-12-13 14:01:00.000,3,DSD,Other,NaN,2024-12-16,Closed,32.707238,-117.079204,4.0,Mobile,NaN,"Our neighbor at 5480 imperial Ave. SAN Diego, ..."
1428597,5038053,2024-12-13 14:01:00.000,615,DSD,Other,NaN,2026-08-20,Closed,32.707238,-117.079204,4.0,Mobile,NaN,"This neighbor at 5480 imperial Ave. SAN Diego,..."
942414,5310443,2025-07-28 21:38:00.000,72,Storm Water Code Enforcement,Other,NaN,2025-10-08,Closed,32.707909,-117.122010,9.0,Mobile,NaN,Illegal dumping from trailer. Dumping waste in...
1513391,5310443,2025-07-28 21:38:00.000,346,Storm Water Code Enforcement,Other,NaN,2026-07-09,Closed,32.707909,-117.122010,9.0,Mobile,NaN,Illegal dumping from trailer. Dumping waste in...
1077335,5352404,2025-08-28 10:42:00.000,39,ESD Complaint/Report,Environmental Services Code Compliance,Container Left Out,2025-10-06,Closed,32.769004,-117.186602,7.0,Mobile,NaN,5958 Lauretta st
1527370,5352404,2025-08-28 10:42:00.000,340,ESD Complaint/Report,Environmental Services Code Compliance,Container Left Out,2026-08-03,Closed,32.768874,-117.186395,7.0,Mobile,NaN,5958 Lauretta st
1114902,5367887,2025-09-08 14:10:00.000,43,ESD Complaint/Report,Waste on Private Property,NaN,2025-10-21,Closed,32.740817,-117.057803,4.0,Mobile,NaN,Appliances on the street in view.
1528053,5367887,2025-09-08 14:10:00.000,324,ESD Complaint/Report,Waste on Private Property,NaN,2026-07-29,Closed,32.740799,-117.057862,4.0,Mobile,NaN,Appliances on the street in view.


In [40]:
df_duplicates.head(16)

,service_request_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,lat,lng,council_district,case_origin,referred,public_description
759134,2416658,2018-10-05 11:54:00,2387,DSD,Development Services - Code Enforcement,Quality of Life Issues,2025-04-18,Closed,32.707764,-117.070610,4.0,Mobile,NaN,"Misc couches, cabinets and random items in fro..."
1411205,2416658,2018-10-05 11:54:01,2952,DSD,Development Services - Code Enforcement,Quality of Life Issues,NaT,New,32.707764,-117.070610,4.0,Mobile,NaN,"Misc couches, cabinets and random items in fro..."
397795,4278707,2023-06-13 07:21:00,482,TSW ROW,Vegetation Encroachment,NaN,2024-10-07,Referred,32.708625,-117.073520,4.0,Mobile,real estate assets,Over grown side walk bushes
1432213,4278707,2023-06-13 07:21:51,1240,TSW,Stormwater,CHANNEL,NaT,In Process,32.708625,-117.073520,4.0,Mobile,real estate assets,Over grown side walk bushes
836731,4600153,2024-01-26 11:02:00,629,TSW ROW,Right-of-Way Code Enforcement,Trenches,2025-10-16,Closed,32.871820,-117.217867,6.0,Mobile,NaN,Subgrade Issue. Pavement lifted 4-6inches. La ...
1435208,4600153,2024-01-26 11:02:24,1013,TSW ROW,Right-of-Way Code Enforcement,Trenches,NaT,In Process,32.871820,-117.217867,6.0,Mobile,NaN,Subgrade Issue. Pavement lifted 4-6inches. La ...
839422,4744770,2024-05-08 16:39:00,342,DSD,Development Services - Code Enforcement,NaN,2025-04-15,Closed,32.699908,-117.097744,4.0,Mobile,NaN,The home next to mine located at 529 S 45th St...
1436947,4744770,2024-05-08 16:39:32,910,DSD,Building and Land Use Enforcement,NaN,NaT,New,32.699908,-117.097744,4.0,Mobile,NaN,The home located at 529 S 45th Street SDCA 921...
525618,4915091,2024-09-04 10:26:00,33,TSW ROW,Vegetation Encroachment,NaN,2024-10-07,Referred,32.708541,-117.073692,4.0,Mobile,real estate assets,Overgrown vegetation
1439021,4915091,2024-09-04 10:26:49,791,TSW,Stormwater,CHANNEL,NaT,In Process,32.708541,-117.073692,4.0,Mobile,real estate assets,Overgrown vegetation
